In [31]:

# ============================================================
# NOTEBOOK 09 — UBIRIS V2 DATASET AUGMENTATION
# Source verification + manifest + dataset statistics
# ============================================================

from pathlib import Path
import json
import pandas as pd
from PIL import Image
from IPython.display import display

PROJECT = Path("/kaggle/working/ubiris_v2_project")
PRE_DIR = PROJECT / "notebook02_preprocessing"
BASELINE_DIR = PROJECT / "notebook04_baseline_recognition"
TRAD_DIR = PROJECT / "notebook05_traditional_augmentation"
GAN_DIR = PROJECT / "notebook07_gan_augmentation"
DIFF_DIR = PROJECT / "notebook06_diffusion_training"
OUT_DIR = PROJECT / "notebook09_dataset_augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REAL_DIR = PRE_DIR / "images_224"
TRAIN_CSV = BASELINE_DIR / "UBIRIS_V2_train_split.csv"
VAL_CSV = BASELINE_DIR / "UBIRIS_V2_validation_split.csv"
TEST_CSV = BASELINE_DIR / "UBIRIS_V2_test_split.csv"
TRAD_META = TRAD_DIR / "UBIRIS_V2_traditional_augmented_metadata.csv"

# ------------------------------------------------------------
# 1. Verify source artifacts
# ------------------------------------------------------------
required = [REAL_DIR, TRAIN_CSV, VAL_CSV, TEST_CSV, TRAD_META]
for path in required:
    assert path.exists(), f"Required source missing: {path}"
    print("FOUND:", path)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
trad_df = pd.read_csv(TRAD_META)

print("\nORIGINAL SPLITS")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Traditional augmentation metadata rows:", len(trad_df))

# ------------------------------------------------------------
# 2. Validate split separation by filename
# ------------------------------------------------------------
def filename_set(df):
    return set(df["filename"].astype(str))

train_names = filename_set(train_df)
val_names = filename_set(val_df)
test_names = filename_set(test_df)

print("\nSPLIT OVERLAP CHECKS")
print("Train ∩ validation:", len(train_names & val_names))
print("Train ∩ test:", len(train_names & test_names))
print("Validation ∩ test:", len(val_names & test_names))

assert not (train_names & val_names)
assert not (train_names & test_names)
assert not (val_names & test_names)

# ------------------------------------------------------------
# 3. Create real training manifest
# ------------------------------------------------------------
real_records = []
missing_real = []

for _, row in train_df.iterrows():
    filename = Path(str(row["filename"])).stem + ".png"
    path = REAL_DIR / filename

    if path.is_file():
        real_records.append({
            "source_type": "real",
            "filename": filename,
            "path": str(path),
            "source_filename": str(row["filename"]),
            "subject": str(row["subject"]),
            "label": int(row["label"]),
            "split": "train",
            "is_synthetic": False
        })
    else:
        missing_real.append(filename)

real_manifest = pd.DataFrame(real_records)
print("\nREAL TRAINING IMAGES FOUND:", len(real_manifest))
print("Missing real training images:", len(missing_real))

# ------------------------------------------------------------
# 4. Validate traditional augmented files and metadata
# ------------------------------------------------------------
trad_image_dir = TRAD_DIR / "augmented_train_images"
trad_records = []
missing_trad = []

for _, row in trad_df.iterrows():
    filename = str(row["augmented_filename"])
    path = trad_image_dir / filename

    if path.is_file():
        trad_records.append({
            "source_type": "traditional_augmented",
            "filename": filename,
            "path": str(path),
            "source_filename": str(row["source_filename"]),
            "subject": str(row["subject"]),
            "label": int(row["label"]),
            "split": "train",
            "is_synthetic": True
        })
    else:
        missing_trad.append(filename)

trad_manifest = pd.DataFrame(trad_records)
print("Traditional augmented files found:", len(trad_manifest))
print("Missing traditional augmented files:", len(missing_trad))

# ------------------------------------------------------------
# 5. GAN/diffusion availability checks
# Do NOT add composite preview grids as training images.
# ------------------------------------------------------------
gan_individual_candidates = []
diff_individual_candidates = []

# Only individual image files from explicitly named folders qualify.
# Preview grids are excluded.
gan_sample_dir = GAN_DIR / "generated_images"
if gan_sample_dir.is_dir():
    gan_individual_candidates = sorted(
        p for p in gan_sample_dir.glob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    )

diff_sample_dir = DIFF_DIR / "generated_images"
if diff_sample_dir.is_dir():
    diff_individual_candidates = sorted(
        p for p in diff_sample_dir.glob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    )

print("\nGENERATIVE SOURCE CHECK")
print("Individual GAN images found:", len(gan_individual_candidates))
print("Individual diffusion images found:", len(diff_individual_candidates))
print("GAN preview grids excluded: yes")
print("Diffusion preview grid excluded: yes")

# No GAN/diffusion images are added unless verified as individual images.
def synthetic_manifest(paths, source_type):
    rows = []
    for path in paths:
        rows.append({
            "source_type": source_type,
            "filename": path.name,
            "path": str(path),
            "source_filename": "",
            "subject": "",
            "label": -1,
            "split": "train",
            "is_synthetic": True
        })
    return pd.DataFrame(rows)

gan_manifest = synthetic_manifest(
    gan_individual_candidates, "gan_synthetic"
)
diff_manifest = synthetic_manifest(
    diff_individual_candidates, "diffusion_synthetic"
)

# ------------------------------------------------------------
# 6. Combine training-only manifest
# ------------------------------------------------------------
manifests = [
    df for df in [
        real_manifest, trad_manifest, gan_manifest, diff_manifest
    ] if not df.empty
]

combined = pd.concat(manifests, ignore_index=True)
MANIFEST_PATH = OUT_DIR / "UBIRIS_V2_augmented_training_manifest.csv"
combined.to_csv(MANIFEST_PATH, index=False)

missing_df = pd.DataFrame(
    [{"source": "real", "filename": x} for x in missing_real] +
    [{"source": "traditional", "filename": x} for x in missing_trad],
    columns=["source", "filename"]
)
MISSING_PATH = OUT_DIR / "UBIRIS_V2_missing_files.csv"
missing_df.to_csv(MISSING_PATH, index=False)

# Source-level counts.
source_counts = (
    combined.groupby(["source_type", "split", "is_synthetic"])
    .size().reset_index(name="image_count")
)
SOURCE_COUNTS_PATH = OUT_DIR / "UBIRIS_V2_augmented_source_counts.csv"
source_counts.to_csv(SOURCE_COUNTS_PATH, index=False)

# Dataset-level summary.
summary = {
    "dataset": "UBIRIS V2",
    "real_train_split_rows": int(len(train_df)),
    "real_training_files_found": int(len(real_manifest)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "traditional_metadata_rows": int(len(trad_df)),
    "traditional_images_found": int(len(trad_manifest)),
    "gan_individual_images_found": int(len(gan_manifest)),
    "diffusion_individual_images_found": int(len(diff_manifest)),
    "combined_training_manifest_rows": int(len(combined)),
    "missing_real_files": int(len(missing_real)),
    "missing_traditional_files": int(len(missing_trad)),
    "validation_and_test_added_to_training_manifest": False,
    "preview_grids_used_as_training_images": False,
    "note": (
        "GAN and diffusion preview grids are excluded. Individual synthetic "
        "images are included only if present in dedicated generated_images folders. "
        "Filename-derived subject labels remain provisional."
    )
}

SUMMARY_PATH = OUT_DIR / "UBIRIS_V2_dataset_augmentation_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print("\nSOURCE COUNTS")
display(source_counts)

print("\nSUMMARY")
print(json.dumps(summary, indent=2))

print("\nOUTPUTS")
for path in sorted(OUT_DIR.iterdir()):
    if path.is_file():
        print(f"OK | {path.name} | {path.stat().st_size:,} bytes")


FOUND: /kaggle/working/ubiris_v2_project/notebook02_preprocessing/images_224
FOUND: /kaggle/working/ubiris_v2_project/notebook04_baseline_recognition/UBIRIS_V2_train_split.csv
FOUND: /kaggle/working/ubiris_v2_project/notebook04_baseline_recognition/UBIRIS_V2_validation_split.csv
FOUND: /kaggle/working/ubiris_v2_project/notebook04_baseline_recognition/UBIRIS_V2_test_split.csv
FOUND: /kaggle/working/ubiris_v2_project/notebook05_traditional_augmentation/UBIRIS_V2_traditional_augmented_metadata.csv

ORIGINAL SPLITS
Train: 4243
Validation: 778
Test: 778
Traditional augmentation metadata rows: 8486

SPLIT OVERLAP CHECKS
Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0

REAL TRAINING IMAGES FOUND: 4243
Missing real training images: 0
Traditional augmented files found: 8486
Missing traditional augmented files: 0

GENERATIVE SOURCE CHECK
Individual GAN images found: 0
Individual diffusion images found: 0
GAN preview grids excluded: yes
Diffusion preview grid excluded: yes

SOURCE COUN

,source_type,split,is_synthetic,image_count
0,real,train,False,4243
1,traditional_augmented,train,True,8486



SUMMARY
{
  "dataset": "UBIRIS V2",
  "real_train_split_rows": 4243,
  "real_training_files_found": 4243,
  "validation_images": 778,
  "test_images": 778,
  "traditional_metadata_rows": 8486,
  "traditional_images_found": 8486,
  "gan_individual_images_found": 0,
  "diffusion_individual_images_found": 0,
  "combined_training_manifest_rows": 12729,
  "missing_real_files": 0,
  "missing_traditional_files": 0,
  "validation_and_test_added_to_training_manifest": false,
  "preview_grids_used_as_training_images": false,
  "note": "GAN and diffusion preview grids are excluded. Individual synthetic images are included only if present in dedicated generated_images folders. Filename-derived subject labels remain provisional."
}

OUTPUTS
OK | UBIRIS_V2_augmented_source_counts.csv | 103 bytes
OK | UBIRIS_V2_augmented_training_manifest.csv | 2,203,709 bytes
OK | UBIRIS_V2_dataset_augmentation_summary.json | 720 bytes
OK | UBIRIS_V2_missing_files.csv | 16 bytes


# Conclusion — Dataset Augmentation on UBIRIS V2

This notebook assembled and verified a training manifest for the UBIRIS V2 ocular-image experiments. The original training split contained 4,243 images, and all 4,243 corresponding preprocessed image files were found. Traditional augmentation metadata contained 8,486 entries, and all 8,486 augmented image files were verified successfully.

The combined manifest contains 12,729 entries: 4,243 real training images and 8,486 traditionally augmented images. This represents three entries per original training image when the original and its two augmented versions are counted together.

Filename overlap checks returned zero overlaps between the original training, validation, and test splits. Validation and test images were not added to the training manifest.

No individual GAN-generated or diffusion-generated images were found in the dedicated output folders checked by this notebook. Consequently, the composite preview grids from the earlier experiments were excluded from the manifest rather than incorrectly treating them as individual training images.

**Limitations:** The GAN and diffusion outputs are not yet integrated into the training dataset. Subject labels are derived from filenames and have not been independently identity-verified. The manifest references existing files and does not create a separate physical copy of the complete dataset.

**Conclusion:** The real and traditional augmentation sources were assembled into a reproducible training manifest with no missing files. This provides the data foundation for the next recognition experiment, while GAN and diffusion integration remains a future step dependent on generating and validating individual synthetic images.

In [32]:

from pathlib import Path
import json
import zipfile
import pandas as pd
from IPython.display import display, FileLink

PROJECT = Path("/kaggle/working/ubiris_v2_project")
OUT_DIR = PROJECT / "notebook09_dataset_augmentation"
ZIP_PATH = Path("/kaggle/working/UBIRIS_V2_Notebook09_Results.zip")

required = [
    OUT_DIR / "UBIRIS_V2_augmented_training_manifest.csv",
    OUT_DIR / "UBIRIS_V2_augmented_source_counts.csv",
    OUT_DIR / "UBIRIS_V2_dataset_augmentation_summary.json",
    OUT_DIR / "UBIRIS_V2_missing_files.csv",
]

for path in required:
    assert path.is_file(), f"Missing artifact: {path}"

manifest = pd.read_csv(required[0])
counts = pd.read_csv(required[1])
summary = json.loads(required[2].read_text())

assert len(manifest) == summary["combined_training_manifest_rows"]
assert summary["missing_real_files"] == 0
assert summary["missing_traditional_files"] == 0
assert not manifest["split"].isin(["validation", "test"]).any()

print("MANIFEST ROWS:", len(manifest))
print("\nSOURCE COUNTS")
print(counts.to_string(index=False))
print("\nSUMMARY")
print(json.dumps(summary, indent=2))

for path in required:
    print(f"OK | {path.name} | {path.stat().st_size:,} bytes")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as zf:
    for path in OUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(PROJECT))

print("\nZIP SIZE:", f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print("ZIP SAVED:", ZIP_PATH)
display(FileLink(str(ZIP_PATH), result_html_prefix="Download Notebook 09 results: "))


MANIFEST ROWS: 12729

SOURCE COUNTS
          source_type split  is_synthetic  image_count
                 real train         False         4243
traditional_augmented train          True         8486

SUMMARY
{
  "dataset": "UBIRIS V2",
  "real_train_split_rows": 4243,
  "real_training_files_found": 4243,
  "validation_images": 778,
  "test_images": 778,
  "traditional_metadata_rows": 8486,
  "traditional_images_found": 8486,
  "gan_individual_images_found": 0,
  "diffusion_individual_images_found": 0,
  "combined_training_manifest_rows": 12729,
  "missing_real_files": 0,
  "missing_traditional_files": 0,
  "validation_and_test_added_to_training_manifest": false,
  "preview_grids_used_as_training_images": false,
  "note": "GAN and diffusion preview grids are excluded. Individual synthetic images are included only if present in dedicated generated_images folders. Filename-derived subject labels remain provisional."
}
OK | UBIRIS_V2_augmented_training_manifest.csv | 2,203,709 bytes
OK |

/kaggle/working/UBIRIS_V2_Notebook09_Results.zip